## Libraries

In [1]:
from scipy.optimize import differential_evolution
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Custom functions
from Functions_carbon_18sep2026_2 import prediction ##### POC ######

## Loading Data

In [3]:
light_path = r"C:\Users\joels\Desktop\ResearchProject\Phytoplankton_Modeling\Doortje\Data\light_1805_1711.txt"
temp_path = r"C:\Users\joels\Desktop\ResearchProject\Phytoplankton_Modeling\Doortje\Data\temp_1805_1711.txt"
wind_path = r"C:\Users\joels\Desktop\ResearchProject\Phytoplankton_Modeling\Doortje\Data\wind_1805_1711.txt"

conc_path = r"C:\Users\joels\Desktop\ResearchProject\Phytoplankton_modeling\Data\AVP2016_concentrations_data.xlsx"
lake_data_path = r"C:\Users\joels\Desktop\ResearchProject\Phytoplankton_Modeling\Data\Verspagen_AmstelveensePoel.xlsx"
DO_path = r"C:\Users\joels\Desktop\ResearchProject\Phytoplankton_Modeling\Results\Dataframes\DO_patterns.xlsx"

light_data = np.loadtxt(light_path)
temp_data = np.loadtxt(temp_path)
wind_data = np.loadtxt(wind_path)

conc_data = pd.read_excel(conc_path)
lake_data = pd.read_excel(lake_data_path, sheet_name="water_parameters")
lake_data2 = pd.read_excel(lake_data_path, sheet_name="HF_WQ")
DO_data = pd.read_excel(DO_path)

In [5]:
conc_data = conc_data[:][0:-3] # Remove NaN rows
lake_data['date'] = pd.to_datetime(                   # Create a column for proper datetime format
    lake_data['year_yyyy'].astype(str) + '-' +
    lake_data['month_mm'].astype(str).str.zfill(2) + '-' +
    lake_data['day_of_month_dd'].astype(str).str.zfill(2) + ' ' +
    lake_data['time_hms'].astype(str),
    format='%Y-%m-%d %H:%M:%S'
)

## Parameters

In [7]:
# time
tst = 60                # Time step in seconds
params = {}  # Initialize parameter dictionary

# General parameters
params["R"] = 8.3144621  # universal gas constant in J/mol K (Jahne et al., 1987)
params["Ea"] = 19.51E3   # activation energy (Jahne et al., 1987), converted from kJ to J
params["A"] = 5.019E-6 * (60 * 60) # scaling constant (Jahne et al., 1987), A is converted from cm2/s to m2/h

# Lake
params["D"] = 0.0027 / 24    # Dilution rate      
params["K_bg"] = 1.3858   # background turbidity in m-1
params["z_max"] = 1.8     # Depth in m  
params['sal'] = 0.521     # salinity in g/L      
# params["CO2_min"] = 3.03982E-06;  # CO2 input from mineralization in mol C/L/h
params["CO2_min"] =0

# phytoplankton
params["N_Chl"] = 6.764E-7   # mol N / ug Chl a
params["P_Chl"] = 1.91E-8    # mol P / ug Chl a
params["S_Chl"] = 1.91E-8    # mol S / ug Chl a
params["C_Chl"] = 4.6E-6     # mol C / ug Chl a

params["PON_POC"] = 0.1405044                       # PON/POC ratio, from carbon_ratios.ipynb
params["POP_POC"] = 0.0038505                       # POP/POC ratio, from carbon_ratios.ipynb
params["S_C"] = params["S_Chl"] / params["C_Chl"]   # S/C ratio

# params["h_i"] = 40     # half saturation constant for light in micromol photons/m2/s
# params["h_i"] = 55.42615437453733
params["h_i"] = 27.602520619602675
params["k"] = 0.0222 / params["C_Chl"]  # specific light attenuation (m2/ mol C)
# params["m"] = 0.1/24   # mortality rate in h-1 
# params["m"] = 0.005009804936307886
params["m"] = 0.0051380757673641295 * 0.5

params["mu_max_cons"] = 3.46E-2           # constant used to determine max growth rate in h-1
params["Q_ten"] = 2.63                    # Change in growth rate for a 10C temperature increase
# params["r_max"] = 2.89E-8                 # maximum respiration rate         
# params["r_max"] = 2.89e-09 / params["C_Chl"]
params["r_max"] = 0.0006282608695652174
# params["v_c_max"] = 7.2E-7                # Maximum carbon uptake in mol C/μg Chl a/h
# params["v_c_max"] = 4.463395875475215e-06 / params["C_Chl"]
params["v_c_max"] = 0.6118895625066884

# params["k_f"] = 8.0E4 / 24   # Converted to L (M m2 h)-1
# params["k_f"] = 3.33333333
params['cf'] = 199313.124328   # calcification factor

params["f_c"] = 0.4          # fraction of maximum C-uptake made up by CO2 uptake
params["h_co2"] = 7.1E-7     # half saturation constant for CO2 [mol/L] (Ji et al, 2020)      
params["h_hco3"] = 8.5E-6    # half saturation constant for HCO3 [mol/L] (Ji et al, 2020)   

# Initialize nutrients and carbon chemistry
params["N_in"]   = 0.2E-3		# initial nitrogen concentration, mol/L
params["P_t_in"] = 5.0E-6       # intial phosphorus concentration, mol/L
params["S_in"]   = 2.0E-5       # Initial sulfur concentration, mol/L
params["Ca_in"]  = 1617.495346E-6 # initial calcium concentration, mol/L

# B = 12.854 * params["C_Chl"]	    # Initial  biomass (mol C/L)
B = 635.2652778 / (10**6)
# d_B = 0	        # change in biomass	###################### Never used ######################33				

params["Alk_in"] = 3.06E-3  # Initial Alkalinity in Eq/L

ppm_CO2 = 400   # pCO2 atmosphere in ppm
params['atm_CO2'] = ppm_CO2 * 1E-6   # pCO2 atmosphere in atm
pH = 8.83                  # Initial pH
params['H_in'] = 10**(-pH)              # Initial hydrogen conc [mol/L or M]
params["DIC_in"] = 3.507E-3             # initial DIC [mol C/L]

oldst_in = [params['DIC_in'], params['Alk_in'], params["N_in"], params["P_t_in"], params["S_in"], B, params["Ca_in"]]

## Main

In [12]:
## Prep
start_date = pd.Timestamp('2016-05-18')

conc_data['hours_since_start'] = ((pd.to_datetime(conc_data['date']) - start_date).dt.total_seconds() / 3600).astype(int)

th_max = len(temp_data) * 24
obs = conc_data[(conc_data['hours_since_start'] >= 0) & (conc_data['hours_since_start'] < th_max)].copy()

obs_indices = obs['hours_since_start'].values
obs_B = obs['POC'].values
obs_DIC = obs['DIC'].values * (10**-6)
obs_pH = obs['pH'].values
pH_sd = obs["pH_sc"].values
obs_Alk = obs['Alk'].values *(10**-6)

mean_obs_B   = np.mean(obs_B)
mean_obs_DIC = np.mean(obs_DIC)
sigma_pH = np.maximum(pH_sd, 0.10)
mean_obs_Alk = np.mean(obs_Alk)

## Weights
w_B   = 1.0
w_DIC = 1.0
w_pH  = 2.0
w_Alk = 1.0

In [14]:
def evo_function(param_vector):
    p = params.copy()

    # p['v_c_max'] = param_vector[0]
    # p['h_i'] = param_vector[1]
    # p['m'] = param_vector[2]
    # p['r_max'] = param_vector[3]
    p['cf'] = param_vector[0]


    oldst = list(oldst_in)

    try:
        df = prediction(temp_data, light_data, wind_data, p, oldst, tst, name_list)
    except Exception:
        return 1e9

    sim_B = df['B'].values
    sim_DIC = df['DIC'].values
    sim_pH = df['pH'].values
    sim_Alk = df['Alk'].values
    
    sim_B_at_obs = sim_B[obs_indices]
    sim_DIC_at_obs = sim_DIC[obs_indices]
    sim_pH_at_obs = sim_pH[obs_indices]
    sim_Alk_at_obs = sim_Alk[obs_indices]

    rmse_B   = np.sqrt(np.mean((sim_B_at_obs - obs_B) ** 2)) / mean_obs_B
    rmse_DIC = np.sqrt(np.mean((sim_DIC_at_obs - obs_DIC) ** 2)) / mean_obs_DIC
    rmse_pH  = np.sqrt(np.mean(((sim_pH_at_obs - obs_pH)/ sigma_pH) ** 2))
    rmse_Alk = np.sqrt(np.mean((sim_Alk_at_obs - obs_Alk) ** 2)) / mean_obs_Alk

    total = w_B * rmse_B + w_DIC * rmse_DIC + w_pH * rmse_pH + w_Alk * rmse_Alk
    
    if not np.isfinite(total):
        return 1e9
    
    return total

In [16]:
bounds = [#(7.2E-8/ params["C_Chl"], 7.2E-6/ params["C_Chl"]),        # v_c_max
          #(20, 80),                # h_i
          #(0.01 / 24, 0.30 / 24),  # m 
          #(2.89E-9/ params["C_Chl"], 2.89E-7/ params["C_Chl"]),  # r_max
          (1.99313124328e4, 1.99313124328e6), #cf
         ]

In [18]:
### Initialize column names for the output dataframe ### 

# WARNING: if you edit this, don't forget to change this too for the prediction function (in the python script) #

name_list = ["B", "d_B",
             "CO2", "HCO3", "CO3",
             "DIC", "pH", "Alk", 
             "N", "P_t", "S",
             "I_in", "I_out", "f_i",
             "t_c",
             "mu", "mu_max", "v_c", "v_co2",
             "CO2_atm", "CO2_min", "DIC_dil",
            ]

In [20]:
result = differential_evolution(evo_function, bounds, seed=8, maxiter=50, popsize=10, disp=True)

differential_evolution step 1: f(x)= 11.017969611330766
Polishing solution with 'L-BFGS-B'


In [21]:
# best_v_c_max = result.x[0]
# best_h_i = result.x[1]
# best_m = result.x[2]
# best_r_max = result.x[3]
# best_cf = result.x[4]

# print("v_c_max: ", best_v_c_max)
# print("h_i: ", best_h_i)
# print("m: ", best_m)
# print("r_max: ", best_r_max)
# print("cf: ", best_cf)

IndexError: index 1 is out of bounds for axis 0 with size 1

In [23]:
print("cf: ", result.x[0])

cf:  1898469.4869896546
